# 01 — Exploring Claude Code GitHub Issues

## Goal
Pull a sample of open issues and inspect the data shape before designing the categorization schema.

## 1. Setup

In [1]:
%pip install -r ../requirements.txt

Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from src.github_client import fetch_issues

REPO = 'anthropics/claude-code'
DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

/Users/kellytaylor/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Fetch a Sample

In [3]:
issues = fetch_issues(REPO, state='open', limit=1000)
print(f'Fetched {len(issues)} issues')

Fetched 1000 issues


## 3. Inspect the Data

In [4]:
# Top-level fields and types
sample = issues[0]
print("Fields:", list(sample.keys()))
print()
for k, v in sample.items():
    print(f"  {k}: {type(v).__name__} — {repr(v)[:80]}")

Fields: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'pinned_comment']

  url: str — 'https://api.github.com/repos/anthropics/claude-code/issues/57169'
  repository_url: str — 'https://api.github.com/repos/anthropics/claude-code'
  labels_url: str — 'https://api.github.com/repos/anthropics/claude-code/issues/57169/labels{/name}'
  comments_url: str — 'https://api.github.com/repos/anthropics/claude-code/issues/57169/comments'
  events_url: str — 'https://api.github.com/repos/anthropics/claude-code/issues/57169/events'
  html_url: str — 'https://github.com/anthropics/claude-

In [5]:
# One full issue — readable view
print(f"#{sample['number']} — {sample['title']}")
print(f"Author : {sample['user']['login']}")
print(f"Labels : {[l['name'] for l in sample['labels']]}")
print(f"Created: {sample['created_at']}")
print(f"Comments: {sample['comments']}")
print()
print("--- Body (first 500 chars) ---")
print((sample["body"] or "")[:500])

#57169 — [BUG] Plan files don't consistently open in review mode with inline comments in VS code extension
Author : sIlENtbuffER
Labels : ['bug', 'area:ide', 'platform:vscode']
Created: 2026-05-08T03:51:56Z
Comments: 1

--- Body (first 500 chars) ---
### Preflight Checklist

- [x] I have searched [existing issues](https://github.com/anthropics/claude-code/issues?q=is%3Aissue%20state%3Aopen%20label%3Abug) and this hasn't been reported yet
- [x] This is a single bug report (please file separate reports for different bugs)
- [x] I am using the latest version of Claude Code

### What's Wrong?

When I manually open a plan file from the VSCode sidebar, it opens as plain markdown without the inline comment capability. The automatic trigger that ope


In [6]:
# Label usage across the sample
from collections import Counter

all_labels = [l["name"] for issue in issues for l in issue["labels"]]
unlabeled = sum(1 for issue in issues if not issue["labels"])

print(f"Unlabeled issues: {unlabeled} / {len(issues)}")
print()
print("Label counts:")
for label, count in Counter(all_labels).most_common():
    print(f"  {count:>3}  {label}")

Unlabeled issues: 23 / 1000

Label counts:
  630  bug
  332  platform:macos
  255  enhancement
  193  platform:windows
  181  has repro
  128  area:tui
  110  duplicate
   93  area:desktop
   91  area:model
   82  area:cost
   81  area:core
   73  platform:vscode
   73  area:mcp
   68  area:skills
   64  platform:linux
   62  area:cowork
   49  area:agents
   48  area:plugins
   47  invalid
   46  area:cli
   44  area:permissions
   43  documentation
   38  area:auth
   36  area:ide
   33  area:docs
   33  area:tools
   33  api:anthropic
   30  area:bash
   30  area:hooks
   28  regression
   26  area:claude-code-web
   23  platform:web
   20  area:sandbox
   20  model
   17  area:ui
   17  area:security
   16  needs-repro
   15  area:statusline
   15  needs-info
   13  data-loss
   13  area:api
   13  memory
   12  area:installation
   12  api:bedrock
   11  area:chrome
   11  area:networking
   11  area:routines
   10  platform:wsl
    9  external
    8  area:browser-extension
    7 

## 4. Save Raw Sample

In [7]:
out = DATA_DIR / "raw_sample.json"
out.write_text(json.dumps(issues, indent=2))
print(f"Saved {len(issues)} issues → {out}")

Saved 1000 issues → ../data/raw_sample.json


## What I Learned

- **Fields available:** 30+ fields from the REST API. Most useful: `number`, `title`, `body`, `labels`, `user`, `comments`, `reactions`, `created_at`, `updated_at`, `html_url`. Note: `type` is always `null`; `reactions.total_count` duplicates the per-reaction sum.
- **Body quality:** Issues follow structured GitHub templates (headers like "Documentation Type", "Documentation Location") — makes body parsing tractable, though older or informal issues may not conform.
- **Label coverage:** All 5 sampled issues were labeled. `enhancement` appeared on all 5; `documentation` on 4 of 5. `area:*` labels are sparse but precise signal when present.
- **Surprises:** Pull requests appear in the `/issues` endpoint and must be filtered out (check for absence of `pull_request` key). The old `gh()` wrapper parsed `--json` fields but never applied filtering.
- **Decisions for categorization schema:** Labels are the primary signal — `area:*` for functional area, `platform:*` for OS/environment, `bug`/`enhancement`/`regression` for type. Title keyword fallback handles unlabeled issues. Priority combines label signals with `reactions + comments` engagement score.